# 01 - Structure Generation 

Builds two silicon structures:
- **Monocrystalline Si** supercell with controlled point defects (vacancies + interstitials) at varying implantation doses
- **Bicrystalline Si** with a single grain boundary at varying misorientation angles

All runs logged to 'defect_db.sqlite'.

In [27]:
import warnings 
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import sqlite3
import random 
from datetime import datetime
from ase.build import bulk
from ase.io import write
from ase import Atoms 
import warnings 


In [19]:
# - Config 
SUPERCELL_SIZE = (10,10,10)
VACANCY_FRACTIONS = [0.01,0.02,0.05,0.10]
ANNEAL_TEMPS = [None,800,900,1000]
MISORIENTATION_ANGLES = [15,30,45]
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Si lattice Parameter 
A = 5.431 # Angstroms

# Supercell face area in cm2 (used for defect density calculations)
WAFER_AREA_CM2 = (A*10*1e-8)**2 # Convert from Angstroms to cm
print(f"Supercell face area: {WAFER_AREA_CM2:.2e} cm^2")


Supercell face area: 2.95e-13 cm^2


In [20]:
def build_monocrystal(vacancy_fraction):
    si = bulk("Si",crystalstructure="diamond",a=A,cubic=True)
    si = si.repeat(SUPERCELL_SIZE)
    num_atoms = len(si)

    # vacancies: remove atoms randomly
    num_vacancies = int(num_atoms * vacancy_fraction)
    vacancy_indices = random.sample(range(num_atoms), num_vacancies)

    # interstitials: displace a subset of remaining atoms off-lattice
    remaining = [i for i in range(num_atoms) if i not in vacancy_indices]
    n_interstitials = max(1, int(num_vacancies * 0.25)) # 25% as many interstitials as vacancies
    interstitial_indices = random.sample(remaining, n_interstitials)

    positions = si.get_positions()
    for idx in interstitial_indices:
        positions[idx] += np.random.uniform(0.5, 1.2, size=3) # Random off-lattice displacement
    si.set_positions(positions)
    
    del si[vacancy_indices] # Remove vacancies
    
    dose = num_vacancies / WAFER_AREA_CM2
    return si, num_vacancies, n_interstitials, dose

print("Monocrystal generation defined")

Monocrystal generation defined


In [21]:
def build_bicrystal(misorientation_deg):
    # grain 1: standard orientation 
    g1 = bulk("Si",crystalstructure="diamond",a=A,cubic=True)
    g1 = g1.repeat((5,10,10)) # 5x10x10 supercell

    # grain 2: rotated around Z axis
    g2 = bulk("Si",crystalstructure="diamond",a=A,cubic=True)
    g2 = g2.repeat((5,10,10))

    angle_rad = np.radians(misorientation_deg)
    rot = np.array([
        [np.cos(angle_rad), -np.sin(angle_rad), 0],
        [np.sin(angle_rad), np.cos(angle_rad), 0],
        [0, 0, 1]
    ])

    positions = g2.get_positions()
    center = positions.mean(axis=0)
    positions = (rot @ (positions - center).T).T + center
    g2.set_positions(positions)

    combined_positions = np.vstack((g1.get_positions(), g2.get_positions()))
    combined_numbers = np.concatenate((g1.get_atomic_numbers(), g2.get_atomic_numbers()))
    new_cell = g1.get_cell().copy()
    new_cell[0,0] *= 2 # Double the cell in x to accommodate 

    bicrystal = Atoms( 
        numbers=combined_numbers,
        positions=combined_positions,
        cell=new_cell,
        pbc=True
    )
    return bicrystal
print("Bicrystal builder defined")

Bicrystal builder defined


In [22]:
conn = sqlite3.connect("defect_db.sqlite")
with open("db_schema.sql") as f:
    conn.executescript(f.read())
print("DB ready")

# - monocrystal runs - 
print("Generating monocrystal structures...")
for vf in VACANCY_FRACTIONS: 
    for temp in ANNEAL_TEMPS:
        si,n_vac,n_int,dose= build_monocrystal(vf)
        frame = f"data/si_mono_vf{int(vf*100):02d}_t{temp or 0}.extxyz"
        write(frame, si)
        defect_density = (n_vac + n_int) / WAFER_AREA_CM2
        conn.execute("""
            INSERT INTO point_defect_runs 
            (run_timestamp,supercell_size,vacancy_fraction,dose_per_cm2,
            anneal_temp_c,vacancy_count,interstitial_count,
            defect_density_cm2,structure_file)
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (
            datetime.now().isoformat(),
            len(si),vf,dose,temp,
            n_vac,n_int,defect_density,frame
        ))
        print(f" vf={vf:.0%} T={temp}C vacancies={n_vac} interstitials={n_int} dose={dose:.2e} defects/cm^2={defect_density:.2e}")
conn.commit()

# - bicrystal runs -
print("\nGenerating bicrystal structures...")
for angle in MISORIENTATION_ANGLES:
    bicrystal = build_bicrystal(angle)
    fname = f"data/si_bi_angle{angle}deg.extxyz"
    write(fname, bicrystal)
    conn.execute("""
        INSERT INTO grain_boundary_runs 
        (run_timestamp,supercell_size,misorientation_angle,
        disordered_fraction,structure_file)
        VALUES (?,?,?,?,?)
    """, (
        datetime.now().isoformat(),
        len(bicrystal),angle,0.0,fname
    ))
    print(f" angle={angle} degrees")
conn.commit()
conn.close()
print("nDone. All structures saved to data/ and logged to defect_db.sqlite")

DB ready
Generating monocrystal structures...
 vf=1% T=NoneC vacancies=80 interstitials=20 dose=2.71e+14 defects/cm^2=3.39e+14
 vf=1% T=800C vacancies=80 interstitials=20 dose=2.71e+14 defects/cm^2=3.39e+14
 vf=1% T=900C vacancies=80 interstitials=20 dose=2.71e+14 defects/cm^2=3.39e+14
 vf=1% T=1000C vacancies=80 interstitials=20 dose=2.71e+14 defects/cm^2=3.39e+14
 vf=2% T=NoneC vacancies=160 interstitials=40 dose=5.42e+14 defects/cm^2=6.78e+14
 vf=2% T=800C vacancies=160 interstitials=40 dose=5.42e+14 defects/cm^2=6.78e+14
 vf=2% T=900C vacancies=160 interstitials=40 dose=5.42e+14 defects/cm^2=6.78e+14
 vf=2% T=1000C vacancies=160 interstitials=40 dose=5.42e+14 defects/cm^2=6.78e+14
 vf=5% T=NoneC vacancies=400 interstitials=100 dose=1.36e+15 defects/cm^2=1.70e+15
 vf=5% T=800C vacancies=400 interstitials=100 dose=1.36e+15 defects/cm^2=1.70e+15
 vf=5% T=900C vacancies=400 interstitials=100 dose=1.36e+15 defects/cm^2=1.70e+15
 vf=5% T=1000C vacancies=400 interstitials=100 dose=1.36e+1

In [28]:
conn = sqlite3.connect("defect_db.sqlite")

# keep only the first 16 runs (the clean ones)
conn.execute("DELETE FROM point_defect_runs WHERE id > 16")
conn.execute("DELETE FROM grain_boundary_runs WHERE id > 3")
conn.commit()

# verify
print(f"point_defect_runs: {pd.read_sql('SELECT COUNT(*) as n FROM point_defect_runs', conn).iloc[0,0]} rows")
print(f"grain_boundary_runs: {pd.read_sql('SELECT COUNT(*) as n FROM grain_boundary_runs', conn).iloc[0,0]} rows")
conn.close()

point_defect_runs: 16 rows
grain_boundary_runs: 3 rows
